# धडा ११ - एजंट-टू-एजंट (A2A) प्रोटोकॉल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## A2A प्रोटोकॉल म्हणजे काय?

**एजंट-टू-एजंट (A2A) प्रोटोकॉल** हा एक खुले मानक आहे ज्यामुळे AI एजंट संवाद साधू शकतात,
एकमेकांना शोधू शकतात, आणि सहकार्य करू शकतात — जरी ते वेगवेगळ्या फ्रेमवर्कवर तयार केलेले असले तरी किंवा वेगवेगळ्या सेवा पुरवठादारांवर होस्ट केले गेले असले तरीही.


मुख्य संकल्पना:

- **शोध** – एजंट आपली क्षमता दर्शवणारा *एजंट कार्ड* प्रकाशित करतात, ज्यामुळे
  इतर एजंट्स (किंवा ऑर्केस्ट्रेटर) साठी योग्य तज्ञ शोधणे सोपे होते.
- **संदेश देवाणघेवाण** – एजंट्स एकसारख्या प्रोटोकॉलद्वारे संरचित संदेशांची देवाणघेवाण करतात, त्यामुळे
  एका एजंटकडून आलेली विनंती दुसऱ्या एजंटकडून त्याच्या अंतर्गत
  रचनेपेक्षा वेगळ्या असूनही समजली जाऊ शकते आणि पूर्ण केली जाऊ शकते.
- **कार्य जीवनचक्र** – A2A मध्ये *सबमिटेड*, *काम करत आहे*, *पूर्ण झाले*, आणि
  *अपयशी* अशा अवस्था परिभाषित केल्या आहेत, ज्यामुळे ऑर्केस्ट्रेटरला नियुक्त काम कसे चालले आहे याची पूर्ण माहिती मिळते.

या धड्यात आम्ही A2A-शैलीतील सहकार्याचे अनुकरण करतो जिथे तीन विशेषीकृत ट्रॅव्हल एजंट्स
अशा वर्कफ्लोमध्ये जोडलेले आहेत जिथे प्रत्येक एजंट त्याचे ज्ञान योगदान देतो आणि पुढील एजंटकडे निकाल पाठवतो.


## विशेष प्रवास एजंट तयार करणे


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## वर्कफ्लो द्वारे मल्टी-एजंट सहकार्य

आम्ही तीन एजंट्सना एक सुसूत्र वर्कफ्लो मध्ये जोडतो जे A2A मेसेज पाठवणीचे प्रतिबिंबित करते:

1. **CurrencyExchangeAgent** वापरकर्त्याचा विनंती प्राप्त करतो आणि चलन मार्गदर्शन तयार करतो.
2. **ActivityPlannerAgent** समृद्ध संदर्भ प्राप्त करतो आणि क्रियाकलाप शिफारसी जोडतो.
3. **TravelManagerAgent** दोन्ही इनपुट संक्षेप करून अंतिम प्रवास सारांश तयार करतो.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## प्रॉडक्शनमध्ये A2A समजून घेणे

प्रॉडक्शन वातावरणात A2A प्रोटोकॉल शक्तिशाली क्रॉस-सर्व्हिस परिदृश्ये अनलॉक करतो:

| क्षमता | वर्णन |
|---|---|
| **क्रास-फ्रेमवर्क इंटरऑप** | एका फ्रेमवर्कने तयार केलेला एजंट कोणत्याही दुसऱ्या A2A-अनुपालन करणार्‍या फ्रेमवर्कने तयार केलेल्या एजंटला कार्ये सोपवू शकतो, जे खरे क्रॉस-संस्था इंटरऑपरेबिलिटी सक्षम करतो. |
| **सेवा सीमा** | एजंट वेगळ्या मायक्रोसर्व्हिसेस, क्लाउड रिजन किंवा अगदी भिन्न संघटनांमध्ये राहू शकतात तरीही सुरळीतपणे सहकार्य करू शकतात. |
| **डायनॅमिक डिस्कव्हरी** | एका ऑर्केस्ट्रेटरला एखाद्या उप-कार्याच्या सर्वोत्तम तज्ज्ञाचा शोध घेण्यासाठी रनटाइमवर एजंट कार्ड रजिस्ट्रीला क्वेरी करता येते. |
| **स्ट्रीमिंग व पुश सूचना** | A2A रिअल-टाइम प्रगती अद्यतनांसाठी सर्व्हर-सेंट ईव्हेंट्स (SSE) आणि दीर्घकालीन कार्यांसाठी पुश सूचना समर्थन करतो. |

वर आपण तयार केलेला वर्कफ्लो हा या पॅटर्नचा एक सोपा, इन-प्रोसेस आवृत्ती आहे. प्रत्यक्ष
तैनातीमध्ये प्रत्येक एजंट HTTP एंडपॉईंट प्रसारित करेल, एजंट कार्ड प्रकाशित करेल आणि
A2A JSON-RPC प्रोटोकॉलद्वारे संवाद साधेल.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
